# AnanthiX AI - Jalon 2 : Exploration PlantVillage

**Objectif** : Analyse du dataset PlantVillage
- 54,303 images
- 38 classes
- 256×256 pixels

In [2]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import tensorflow_datasets as tfds
import tensorflow as tf

# Charger PlantVillage depuis TensorFlow Datasets
print("Téléchargement PlantVillage")
ds, info = tfds.load('plant_village', with_info=True, as_supervised=True)

print(f"\n Dataset info :")
print(f"  - Nombre total d'images : {info.splits['train'].num_examples}")
print(f"  - Classes : {info.features['label'].num_classes}")
print(f"  - Split : {list(info.splits.keys())}")

# Extraire les métadonnées
total_images = info.splits['train'].num_examples
num_classes = info.features['label'].num_classes
class_names = info.features['label'].names

print(f"\n Classes ({num_classes}) :")
for idx, name in enumerate(class_names[:5]):
    print(f"  {idx}: {name}")
print(f"  ... ({num_classes - 5} autres classes)")

# Vérifier structure
train_ds = ds['train']
sample = next(iter(train_ds.take(1)))
image, label = sample
print(f"\n Sample shape : Image {image.shape}, Label {label.numpy()}")

Téléchargement PlantVillage


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.F0U0BU_1.0.2/plant_village-train.tfrecord*...:   …

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.

 Dataset info :
  - Nombre total d'images : 54303
  - Classes : 38
  - Split : [Split('train')]

 Classes (38) :
  0: Apple___Apple_scab
  1: Apple___Black_rot
  2: Apple___Cedar_apple_rust
  3: Apple___healthy
  4: Blueberry___healthy
  ... (33 autres classes)

 Sample shape : Image (256, 256, 3), Label 35


In [3]:
# Compter les images par classe
print("\nComptage des images par classe")
class_counts = Counter()
total_processed = 0

for image, label in train_ds:
    class_idx = label.numpy()
    class_name = class_names[class_idx]
    class_counts[class_name] += 1
    total_processed += 1

    if total_processed % 10000 == 0:
        print(f"  Traité : {total_processed}/{total_images}")

print(f"\n Total images comptées : {sum(class_counts.values())}")
print(f"\n Top 10 classes (par nombre d'images) :")
for class_name, count in class_counts.most_common(10):
    print(f"  {class_name}: {count}")

# Métriques déséquilibre
counts_list = list(class_counts.values())
min_count = min(counts_list)
max_count = max(counts_list)
mean_count = np.mean(counts_list)
std_count = np.std(counts_list)

imbalance_ratio = max_count / min_count

print(f"\n Statistiques déséquilibre :")
print(f"  - Min images/classe : {min_count}")
print(f"  - Max images/classe : {max_count}")
print(f"  - Moyenne : {mean_count:.1f}")
print(f"  - Écart-type : {std_count:.1f}")
print(f"  - Ratio déséquilibre (max/min) : {imbalance_ratio:.1f}x")


Comptage des images par classe
  Traité : 10000/54303
  Traité : 20000/54303
  Traité : 30000/54303
  Traité : 40000/54303
  Traité : 50000/54303

 Total images comptées : 54303

 Top 10 classes (par nombre d'images) :
  Orange___Haunglongbing_(Citrus_greening): 5507
  Tomato___Tomato_Yellow_Leaf_Curl_Virus: 5357
  Soybean___healthy: 5090
  Peach___Bacterial_spot: 2297
  Tomato___Bacterial_spot: 2127
  Tomato___Late_blight: 1908
  Squash___Powdery_mildew: 1835
  Tomato___Septoria_leaf_spot: 1771
  Tomato___Spider_mites Two-spotted_spider_mite: 1676
  Apple___healthy: 1645

 Statistiques déséquilibre :
  - Min images/classe : 152
  - Max images/classe : 5507
  - Moyenne : 1429.0
  - Écart-type : 1254.9
  - Ratio déséquilibre (max/min) : 36.2x


In [4]:
# Analyser résolutions
print("\nAnalyse des résolutions")
resolutions = Counter()
sample_size = 5000  # Prendre un échantillon pour vitesse
total_analyzed = 0

for image, label in train_ds.take(sample_size):
    h, w = image.shape[0], image.shape[1]
    resolutions[f"{h}x{w}"] += 1
    total_analyzed += 1

print(f" Résolutions trouvées ({len(resolutions)} uniques) :")
for res, count in resolutions.most_common(5):
    percent = 100 * count / total_analyzed
    print(f"  {res}: {count} images ({percent:.1f}%)")

# Analyser valeurs pixels
print("\nAnalyse valeurs pixels")
pixel_mins, pixel_maxs, pixel_means = [], [], []

for image, _ in train_ds.take(500):
    img_array = image.numpy().astype(np.float32)
    pixel_mins.append(img_array.min())
    pixel_maxs.append(img_array.max())
    pixel_means.append(img_array.mean())

print(f" Valeurs pixels :")
print(f"  - Min global : {np.min(pixel_mins):.2f}")
print(f"  - Max global : {np.max(pixel_maxs):.2f}")
print(f"  - Moyenne globale : {np.mean(pixel_means):.2f}")


Analyse des résolutions
 Résolutions trouvées (1 uniques) :
  256x256: 5000 images (100.0%)

Analyse valeurs pixels
 Valeurs pixels :
  - Min global : 0.00
  - Max global : 255.00
  - Moyenne globale : 114.95


In [7]:
from pathlib import Path
import json
import numpy as np

BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
DATA_PATH = BASE_PATH / 'data'

# IMPORTANT : créer le dossier
DATA_PATH.mkdir(parents=True, exist_ok=True)

# Créer dictionnaire métadonnées
metadata = {
    "dataset_name": "PlantVillage",
    "total_images": int(total_images),
    "num_classes": int(num_classes),
    "class_names": class_names,
    "class_counts": {name: int(count) for name, count in class_counts.items()},
    "imbalance_stats": {
        "min_count": int(min_count),
        "max_count": int(max_count),
        "mean_count": float(mean_count),
        "std_count": float(std_count),
        "imbalance_ratio": float(imbalance_ratio)
    },
    "resolution_stats": {
        "sample_size": sample_size,
        "resolutions": {res: int(count) for res, count in resolutions.items()}
    },
    "pixel_stats": {
        "min": float(np.min(pixel_mins)),
        "max": float(np.max(pixel_maxs)),
        "mean": float(np.mean(pixel_means))
    }
}

# Sauvegarder
metadata_path = DATA_PATH / 'plantvillage_metadata.json'

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n Métadonnées sauvegardées : {metadata_path}")


 Métadonnées sauvegardées : /content/drive/MyDrive/AnanthiX_AI/data/plantvillage_metadata.json


In [9]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')

DATA_PATH = BASE_PATH / 'data'
RESULTS_PATH = BASE_PATH / 'results'

DATA_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print("\nGénération grille d'exemples")

fig, axes = plt.subplots(10, 4, figsize=(16, 20))
fig.suptitle('PlantVillage : Exemples par classe', fontsize=16, fontweight='bold')

axes = axes.flatten()
class_examples_shown = {name: 0 for name in class_names}

example_idx = 0
for image, label in train_ds:
    if example_idx >= 40:
        break

    label_idx = label.numpy()
    class_name = class_names[label_idx]

    if class_examples_shown[class_name] < 1:
        ax = axes[example_idx]
        img_array = image.numpy().astype(np.uint8)

        ax.imshow(img_array)
        ax.set_title(class_name, fontsize=8)
        ax.axis('off')

        class_examples_shown[class_name] += 1
        example_idx += 1

plt.tight_layout()

viz_path = RESULTS_PATH / 'exploration_samples.png'
plt.savefig(viz_path, dpi=150, bbox_inches='tight')

print(f" Visualisation sauvegardée : {viz_path}")

plt.close()


Génération grille d'exemples
 Visualisation sauvegardée : /content/drive/MyDrive/AnanthiX_AI/results/exploration_samples.png


In [10]:
# Histogramme distribution classes
print("\nGénération histogramme distribution.")

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 1. Distribution complète (tronquée pour lisibilité)
sorted_counts = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
class_names_sorted = [name.replace('_', '\n') for name, _ in sorted_counts]
counts_sorted = [count for _, count in sorted_counts]

ax1 = axes[0]
bars1 = ax1.bar(range(len(class_names_sorted)), counts_sorted, color='steelblue', alpha=0.7)
ax1.set_xlabel('Classe', fontweight='bold')
ax1.set_ylabel('Nombre d\'images', fontweight='bold')
ax1.set_title('Distribution PlantVillage (38 classes)', fontweight='bold', fontsize=12)
ax1.set_xticks(range(len(class_names_sorted)))
ax1.set_xticklabels(class_names_sorted, rotation=45, ha='right', fontsize=7)
ax1.grid(axis='y', alpha=0.3)

# 2. Histogramme des counts (distribution de distribution)
ax2 = axes[1]
ax2.hist(counts_sorted, bins=20, color='coral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Nombre d\'images par classe', fontweight='bold')
ax2.set_ylabel('Fréquence (nb de classes)', fontweight='bold')
ax2.set_title('Distribution du déséquilibre', fontweight='bold', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
dist_path = RESULTS_PATH / 'exploration_distribution.png'
plt.savefig(dist_path, dpi=150, bbox_inches='tight')
print(f" Histogramme sauvegardé : {dist_path}")
plt.close()


Génération histogramme distribution.
 Histogramme sauvegardé : /content/drive/MyDrive/AnanthiX_AI/results/exploration_distribution.png


In [13]:
import os
import shutil
from pathlib import Path
from google.colab import drive

# Étape 1 : Nettoyer le mountpoint existant
mountpoint = Path('/content/drive')

if mountpoint.exists():
    print(f"Nettoyage de {mountpoint}...")
    try:
        shutil.rmtree(mountpoint)
        print(f"✓ {mountpoint} supprimé")
    except Exception as e:
        print(f"✗ Erreur suppression : {e}")
        print("Tentative alternative...")
        os.system(f'rm -rf {mountpoint}')

# Étape 2 : Remonter Google Drive
print("\nRemontage Google Drive...")
drive.mount('/content/drive')

print("✓ Google Drive remonté avec succès\n")

# Étape 3 : Vérifier structure créée
BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')

print("="*60)
print("VÉRIFICATION STRUCTURE ANANTHIX_AI")
print("="*60)

# Lister tous les fichiers récursivement
if BASE_PATH.exists():
    print(f"\n✓ Base path existe : {BASE_PATH}\n")

    for root, dirs, files in os.walk(BASE_PATH):
        level = root.replace(str(BASE_PATH), '').count(os.sep)
        indent = ' ' * 2 * level
        folder_name = os.path.basename(root) if root != str(BASE_PATH) else 'AnanthiX_AI'
        print(f'{indent}{folder_name}/')

        subindent = ' ' * 2 * (level + 1)
        for file in sorted(files):
            filepath = Path(root) / file
            try:
                size_kb = filepath.stat().st_size / 1024
                print(f'{subindent}{file} ({size_kb:.1f} KB)')
            except:
                print(f'{subindent}{file} (erreur lecture)')

    print("\n" + "="*60)
    print("RÉSUMÉ")
    print("="*60)

    # Compter fichiers
    file_count = sum(len(files) for _, _, files in os.walk(BASE_PATH))
    print(f"✓ Total fichiers : {file_count}")

else:
    print(f"\n✗ Base path n'existe pas : {BASE_PATH}")
    print("\nCréation de la structure...")

    for folder in ['data', 'results', 'models', 'notebooks']:
        path = BASE_PATH / folder
        path.mkdir(parents=True, exist_ok=True)
        print(f"✓ {folder}/ créé")

Nettoyage de /content/drive...
✓ /content/drive supprimé

Remontage Google Drive...
Mounted at /content/drive
✓ Google Drive remonté avec succès

VÉRIFICATION STRUCTURE ANANTHIX_AI

✓ Base path existe : /content/drive/MyDrive/AnanthiX_AI

AnanthiX_AI/
  data/
  results/
  models/
  notebooks/
    01_exploration_plantvillage.ipynb (65.6 KB)

RÉSUMÉ
✓ Total fichiers : 1


In [14]:
import json
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import tensorflow_datasets as tfds

print("SAUVEGARDE MANUELLE DES RÉSULTATS ÉTAPE 1")

# Chemins Google Drive
BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
DATA_PATH = BASE_PATH / 'data'
RESULTS_PATH = BASE_PATH / 'results'

# Recréer les dossiers
for path in [DATA_PATH, RESULTS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print(f"\n Dossiers créés")


# ÉTAPE 1 : Recharger les données depuis cache TensorFlow
print("\nRecherche du dataset en cache TensorFlow.")

ds, info = tfds.load('plant_village', with_info=True, as_supervised=True)
train_ds = ds['train']
total_images = info.splits['train'].num_examples
num_classes = info.features['label'].num_classes
class_names = info.features['label'].names

print(f"Dataset chargé : {total_images} images, {num_classes} classes")

# ÉTAPE 2 : Recomputer les statistiques

print("\nRecalcul des statistiques.")

class_counts = Counter()
for image, label in train_ds:
    class_idx = label.numpy()
    class_name = class_names[class_idx]
    class_counts[class_name] += 1

counts_list = list(class_counts.values())
min_count = min(counts_list)
max_count = max(counts_list)
mean_count = np.mean(counts_list)
std_count = np.std(counts_list)
imbalance_ratio = max_count / min_count

print(f" Statistiques calculées")

# ÉTAPE 3 : Créer metadata JSON

print("\nCréation metadata.json.")

metadata = {
    "dataset_name": "PlantVillage",
    "total_images": int(total_images),
    "num_classes": int(num_classes),
    "class_names": class_names,
    "class_counts": {name: int(count) for name, count in class_counts.items()},
    "imbalance_stats": {
        "min_count": int(min_count),
        "max_count": int(max_count),
        "mean_count": float(mean_count),
        "std_count": float(std_count),
        "imbalance_ratio": float(imbalance_ratio)
    }
}

metadata_path = DATA_PATH / 'plantvillage_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f" Sauvegardé : {metadata_path}")
assert metadata_path.exists(), "Erreur : metadata.json non créé"

# ÉTAPE 4 : Créer visualisation exemples

print("\nCréation visualisation exemples")

fig, axes = plt.subplots(10, 4, figsize=(16, 20))
fig.suptitle('PlantVillage : Exemples par classe', fontsize=16, fontweight='bold')

axes = axes.flatten()
class_examples_shown = {name: 0 for name in class_names}

example_idx = 0
for image, label in train_ds:
    if example_idx >= 40:
        break

    label_idx = label.numpy()
    class_name = class_names[label_idx]

    if class_examples_shown[class_name] < 1:
        ax = axes[example_idx]
        img_array = image.numpy().astype(np.uint8)
        ax.imshow(img_array)
        ax.set_title(class_name, fontsize=8)
        ax.axis('off')

        class_examples_shown[class_name] += 1
        example_idx += 1

plt.tight_layout()
viz_path = RESULTS_PATH / 'exploration_samples.png'
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.close()

print(f" Sauvegardé : {viz_path}")
assert viz_path.exists(), "Erreur : exploration_samples.png non créé"

# ÉTAPE 5 : Créer histogramme distribution

print("\nCréation histogramme distribution")

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

sorted_counts = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
class_names_sorted = [name.replace('_', '\n') for name, _ in sorted_counts]
counts_sorted = [count for _, count in sorted_counts]

ax1 = axes[0]
ax1.bar(range(len(class_names_sorted)), counts_sorted, color='steelblue', alpha=0.7)
ax1.set_xlabel('Classe', fontweight='bold')
ax1.set_ylabel('Nombre d\'images', fontweight='bold')
ax1.set_title('Distribution PlantVillage (38 classes)', fontweight='bold', fontsize=12)
ax1.set_xticks(range(len(class_names_sorted)))
ax1.set_xticklabels(class_names_sorted, rotation=45, ha='right', fontsize=7)
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.hist(counts_sorted, bins=20, color='coral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Nombre d\'images par classe', fontweight='bold')
ax2.set_ylabel('Fréquence (nb de classes)', fontweight='bold')
ax2.set_title('Distribution du déséquilibre', fontweight='bold', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
dist_path = RESULTS_PATH / 'exploration_distribution.png'
plt.savefig(dist_path, dpi=150, bbox_inches='tight')
plt.close()

print(f" Sauvegardé : {dist_path}")
assert dist_path.exists(), "Erreur : exploration_distribution.png non créé"

# ÉTAPE 6 : Vérification finale


print("VÉRIFICATION SAUVEGARDE")

files_to_check = [
    ("Metadata", metadata_path),
    ("Exemples", viz_path),
    ("Distribution", dist_path)
]

for name, path in files_to_check:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f" {name:15} : {path.name:40} ({size_kb:8.1f} KB)")
    else:
        print(f" {name:15} : MANQUANT")


print("RÉSUMÉ ÉTAPE 1")

print(f"\nDataset:")
print(f"  • Total images: {metadata['total_images']:,}")
print(f"  • Nombre de classes: {metadata['num_classes']}")
print(f"\nDéséquilibre:")
print(f"  • Classe la plus grande: {max_count} images")
print(f"  • Classe la plus petite: {min_count} images")
print(f"  • Ratio: {imbalance_ratio:.1f}x")
print(f"\nFichiers sauvegardés dans Google Drive:")
print(f"   {DATA_PATH.relative_to(BASE_PATH)}/plantvillage_metadata.json")
print(f"   {RESULTS_PATH.relative_to(BASE_PATH)}/exploration_samples.png")
print(f"  {RESULTS_PATH.relative_to(BASE_PATH)}/exploration_distribution.png")

print("\n Étape 1 complétée et sauvegardée avec succès")

SAUVEGARDE MANUELLE DES RÉSULTATS ÉTAPE 1

 Dossiers créés

Recherche du dataset en cache TensorFlow.
Dataset chargé : 54303 images, 38 classes

Recalcul des statistiques.
 Statistiques calculées

Création metadata.json.
 Sauvegardé : /content/drive/MyDrive/AnanthiX_AI/data/plantvillage_metadata.json

Création visualisation exemples
 Sauvegardé : /content/drive/MyDrive/AnanthiX_AI/results/exploration_samples.png

Création histogramme distribution
 Sauvegardé : /content/drive/MyDrive/AnanthiX_AI/results/exploration_distribution.png
VÉRIFICATION SAUVEGARDE
 Metadata        : plantvillage_metadata.json               (     2.8 KB)
 Exemples        : exploration_samples.png                  (  4677.3 KB)
 Distribution    : exploration_distribution.png             (   173.2 KB)
RÉSUMÉ ÉTAPE 1

Dataset:
  • Total images: 54,303
  • Nombre de classes: 38

Déséquilibre:
  • Classe la plus grande: 5507 images
  • Classe la plus petite: 152 images
  • Ratio: 36.2x

Fichiers sauvegardés dans Googl

In [15]:
import json
from pathlib import Path

# Créer le contenu du notebook 02
notebook_content = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# AnanthiX AI - Jalon 2 : Preprocessing + Training Baseline\n",
                "\n",
                "**Objectif** : Préparer les données et entraîner ResNet-50 baseline\n",
                "\n",
                "## Étapes\n",
                "1. Charger metadata depuis étape 1\n",
                "2. Créer splits train/val/test (70/15/15)\n",
                "3. Appliquer normalisation ImageNet\n",
                "4. Augmentation données (train uniquement)\n",
                "5. Entraîner ResNet-50 baseline (10 epochs)\n",
                "6. Calculer métriques\n",
                "7. Sauvegarder modèle + résultats"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Setup : Imports + Google Drive"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import os\n",
                "import json\n",
                "import numpy as np\n",
                "import matplotlib.pyplot as plt\n",
                "import seaborn as sns\n",
                "from pathlib import Path\n",
                "from collections import Counter\n",
                "import pickle\n",
                "\n",
                "import torch\n",
                "import torch.nn as nn\n",
                "import torch.optim as optim\n",
                "from torch.utils.data import Dataset, DataLoader\n",
                "import torchvision.transforms as transforms\n",
                "import torchvision.models as models\n",
                "\n",
                "import tensorflow_datasets as tfds\n",
                "from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score\n",
                "\n",
                "from google.colab import drive\n",
                "\n",
                "# Vérifier GPU\n",
                "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
                "print(f'Device: {device}')\n",
                "if torch.cuda.is_available():\n",
                "    print(f'GPU: {torch.cuda.get_device_name(0)}')\n",
                "    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')\n",
                "\n",
                "# Monter Google Drive\n",
                "drive.mount('/content/drive', force_remount=False)\n",
                "\n",
                "# Chemins\n",
                "BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')\n",
                "DATA_PATH = BASE_PATH / 'data'\n",
                "RESULTS_PATH = BASE_PATH / 'results'\n",
                "MODELS_PATH = BASE_PATH / 'models'\n",
                "\n",
                "for path in [DATA_PATH, RESULTS_PATH, MODELS_PATH]:\n",
                "    path.mkdir(parents=True, exist_ok=True)\n",
                "\n",
                "print(f'✓ Base path: {BASE_PATH}')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Charger dataset et metadata étape 1"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Charger metadata sauvegardée étape 1\n",
                "metadata_path = DATA_PATH / 'plantvillage_metadata.json'\n",
                "with open(metadata_path, 'r') as f:\n",
                "    metadata = json.load(f)\n",
                "\n",
                "class_names = metadata['class_names']\n",
                "num_classes = len(class_names)\n",
                "total_images = metadata['total_images']\n",
                "\n",
                "print(f'✓ Metadata chargée')\n",
                "print(f'  - Total images: {total_images}')\n",
                "print(f'  - Classes: {num_classes}')\n",
                "\n",
                "# Charger dataset TensorFlow\n",
                "print('\\nChargement PlantVillage...')\n",
                "ds, info = tfds.load('plant_village', with_info=True, as_supervised=True)\n",
                "train_ds = ds['train']\n",
                "print(f'✓ Dataset chargé')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Créer splits train/val/test"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "print('Création splits train/val/test...')\n",
                "\n",
                "# Créer indices par classe\n",
                "class_indices = {name: [] for name in class_names}\n",
                "idx = 0\n",
                "\n",
                "for image, label in train_ds:\n",
                "    label_idx = label.numpy()\n",
                "    class_name = class_names[label_idx]\n",
                "    class_indices[class_name].append(idx)\n",
                "    idx += 1\n",
                "\n",
                "print(f'✓ Indices par classe créés')\n",
                "\n",
                "# Split 70/15/15 par classe (stratifié)\n",
                "train_indices = []\n",
                "val_indices = []\n",
                "test_indices = []\n",
                "\n",
                "for class_name, indices in class_indices.items():\n",
                "    n = len(indices)\n",
                "    n_train = int(0.7 * n)\n",
                "    n_val = int(0.15 * n)\n",
                "    \n",
                "    train_indices.extend(indices[:n_train])\n",
                "    val_indices.extend(indices[n_train:n_train + n_val])\n",
                "    test_indices.extend(indices[n_train + n_val:])\n",
                "\n",
                "print(f'✓ Splits créés:')\n",
                "print(f'  - Train: {len(train_indices)} ({100*len(train_indices)/total_images:.1f}%)')\n",
                "print(f'  - Val:   {len(val_indices)} ({100*len(val_indices)/total_images:.1f}%)')\n",
                "print(f'  - Test:  {len(test_indices)} ({100*len(test_indices)/total_images:.1f}%)')\n",
                "\n",
                "# Sauvegarder splits\n",
                "splits = {\n",
                "    'train': train_indices,\n",
                "    'val': val_indices,\n",
                "    'test': test_indices\n",
                "}\n",
                "\n",
                "splits_path = DATA_PATH / 'splits.pkl'\n",
                "with open(splits_path, 'wb') as f:\n",
                "    pickle.dump(splits, f)\n",
                "\n",
                "print(f'\\n✓ Splits sauvegardés: {splits_path}')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Créer PyTorch Dataset avec augmentation"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "class PlantVillageDataset(Dataset):\n",
                "    def __init__(self, dataset, indices, class_names, transform=None):\n",
                "        self.dataset = dataset\n",
                "        self.indices = indices\n",
                "        self.class_names = class_names\n",
                "        self.transform = transform\n",
                "        \n",
                "    def __len__(self):\n",
                "        return len(self.indices)\n",
                "    \n",
                "    def __getitem__(self, idx):\n",
                "        data_idx = self.indices[idx]\n",
                "        \n",
                "        # Charger depuis dataset TensorFlow\n",
                "        for i, (image, label) in enumerate(self.dataset):\n",
                "            if i == data_idx:\n",
                "                image = image.numpy().astype(np.uint8)\n",
                "                label_idx = label.numpy()\n",
                "                \n",
                "                if self.transform:\n",
                "                    image = self.transform(image)\n",
                "                else:\n",
                "                    image = transforms.ToTensor()(image)\n",
                "                \n",
                "                return image, label_idx\n",
                "        \n",
                "        raise IndexError(f'Index {data_idx} not found')\n",
                "\n",
                "# Transformations\n",
                "imagenet_mean = [0.485, 0.456, 0.406]\n",
                "imagenet_std = [0.229, 0.224, 0.225]\n",
                "\n",
                "train_transform = transforms.Compose([\n",
                "    transforms.ToPILImage(),\n",
                "    transforms.RandomRotation(15),\n",
                "    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),\n",
                "    transforms.ColorJitter(brightness=0.1, contrast=0.1),\n",
                "    transforms.RandomHorizontalFlip(p=0.5),\n",
                "    transforms.RandomCrop(256, pad_if_needed=True),\n",
                "    transforms.ToTensor(),\n",
                "    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)\n",
                "])\n",
                "\n",
                "val_test_transform = transforms.Compose([\n",
                "    transforms.ToPILImage(),\n",
                "    transforms.ToTensor(),\n",
                "    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)\n",
                "])\n",
                "\n",
                "print('✓ Transformations créées')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Créer DataLoaders"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "print('Création DataLoaders...')\n",
                "\n",
                "batch_size = 32\n",
                "num_workers = 0  # Colab n'aime pas les workers\n",
                "\n",
                "train_dataset = PlantVillageDataset(train_ds, train_indices, class_names, transform=train_transform)\n",
                "val_dataset = PlantVillageDataset(train_ds, val_indices, class_names, transform=val_test_transform)\n",
                "test_dataset = PlantVillageDataset(train_ds, test_indices, class_names, transform=val_test_transform)\n",
                "\n",
                "train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)\n",
                "val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n",
                "test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n",
                "\n",
                "print(f'✓ DataLoaders créés:')\n",
                "print(f'  - Train batches: {len(train_loader)}')\n",
                "print(f'  - Val batches: {len(val_loader)}')\n",
                "print(f'  - Test batches: {len(test_loader)}')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Charger modèle ResNet-50"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "print('Chargement ResNet-50 pré-entraîné...')\n",
                "\n",
                "model = models.resnet50(pretrained=True)\n",
                "\n",
                "# Adapter dernière couche\n",
                "model.fc = nn.Linear(2048, num_classes)\n",
                "\n",
                "# Freezer early layers\n",
                "for param in model.layer1.parameters():\n",
                "    param.requires_grad = False\n",
                "for param in model.layer2.parameters():\n",
                "    param.requires_grad = False\n",
                "\n",
                "model = model.to(device)\n",
                "\n",
                "# Compter paramètres\n",
                "total_params = sum(p.numel() for p in model.parameters())\n",
                "trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n",
                "\n",
                "print(f'✓ Modèle chargé:')\n",
                "print(f'  - Total params: {total_params:,}')\n",
                "print(f'  - Trainable: {trainable_params:,}')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Configurer entraînement"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "criterion = nn.CrossEntropyLoss()\n",
                "optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)\n",
                "scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)\n",
                "\n",
                "num_epochs = 10\n",
                "early_stopping_patience = 3\n",
                "\n",
                "print('✓ Configuration complète')\n",
                "print(f'  - Loss: CrossEntropyLoss')\n",
                "print(f'  - Optimizer: Adam (lr=1e-4)')\n",
                "print(f'  - Scheduler: ReduceLROnPlateau')\n",
                "print(f'  - Epochs: {num_epochs}')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Boucle d'entraînement"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "print('Démarrage entraînement...\\n')\n",
                "\n",
                "history = {\n",
                "    'train_loss': [],\n",
                "    'train_acc': [],\n",
                "    'val_loss': [],\n",
                "    'val_acc': []\n",
                "}\n",
                "\n",
                "best_val_loss = float('inf')\n",
                "patience_counter = 0\n",
                "\n",
                "for epoch in range(num_epochs):\n",
                "    # Training\n",
                "    model.train()\n",
                "    train_loss = 0.0\n",
                "    train_correct = 0\n",
                "    train_total = 0\n",
                "    \n",
                "    for batch_idx, (images, labels) in enumerate(train_loader):\n",
                "        images = images.to(device)\n",
                "        labels = torch.tensor(labels).long().to(device)\n",
                "        \n",
                "        optimizer.zero_grad()\n",
                "        outputs = model(images)\n",
                "        loss = criterion(outputs, labels)\n",
                "        \n",
                "        loss.backward()\n",
                "        optimizer.step()\n",
                "        \n",
                "        train_loss += loss.item()\n",
                "        _, predicted = torch.max(outputs.data, 1)\n",
                "        train_total += labels.size(0)\n",
                "        train_correct += (predicted == labels).sum().item()\n",
                "        \n",
                "        if (batch_idx + 1) % 50 == 0:\n",
                "            print(f'  Batch {batch_idx + 1}/{len(train_loader)}')\n",
                "    \n",
                "    train_loss = train_loss / len(train_loader)\n",
                "    train_acc = 100 * train_correct / train_total\n",
                "    \n",
                "    # Validation\n",
                "    model.eval()\n",
                "    val_loss = 0.0\n",
                "    val_correct = 0\n",
                "    val_total = 0\n",
                "    \n",
                "    with torch.no_grad():\n",
                "        for images, labels in val_loader:\n",
                "            images = images.to(device)\n",
                "            labels = torch.tensor(labels).long().to(device)\n",
                "            \n",
                "            outputs = model(images)\n",
                "            loss = criterion(outputs, labels)\n",
                "            \n",
                "            val_loss += loss.item()\n",
                "            _, predicted = torch.max(outputs.data, 1)\n",
                "            val_total += labels.size(0)\n",
                "            val_correct += (predicted == labels).sum().item()\n",
                "    \n",
                "    val_loss = val_loss / len(val_loader)\n",
                "    val_acc = 100 * val_correct / val_total\n",
                "    \n",
                "    # Sauvegarder historique\n",
                "    history['train_loss'].append(train_loss)\n",
                "    history['train_acc'].append(train_acc)\n",
                "    history['val_loss'].append(val_loss)\n",
                "    history['val_acc'].append(val_acc)\n",
                "    \n",
                "    # Early stopping\n",
                "    scheduler.step(val_loss)\n",
                "    \n",
                "    if val_loss < best_val_loss:\n",
                "        best_val_loss = val_loss\n",
                "        patience_counter = 0\n",
                "    else:\n",
                "        patience_counter += 1\n",
                "    \n",
                "    print(f'Epoch {epoch+1}/{num_epochs}')\n",
                "    print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%')\n",
                "    print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%')\n",
                "    \n",
                "    if patience_counter >= early_stopping_patience:\n",
                "        print(f'Early stopping at epoch {epoch+1}')\n",
                "        break\n",
                "\n",
                "print('\\n✓ Entraînement complété')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Évaluation sur test set"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "print('Évaluation sur test set...')\n",
                "\n",
                "model.eval()\n",
                "test_correct = 0\n",
                "test_total = 0\n",
                "all_predictions = []\n",
                "all_labels = []\n",
                "\n",
                "with torch.no_grad():\n",
                "    for images, labels in test_loader:\n",
                "        images = images.to(device)\n",
                "        labels = torch.tensor(labels).long().to(device)\n",
                "        \n",
                "        outputs = model(images)\n",
                "        _, predicted = torch.max(outputs.data, 1)\n",
                "        \n",
                "        test_total += labels.size(0)\n",
                "        test_correct += (predicted == labels).sum().item()\n",
                "        \n",
                "        all_predictions.extend(predicted.cpu().numpy())\n",
                "        all_labels.extend(labels.cpu().numpy())\n",
                "\n",
                "test_acc = 100 * test_correct / test_total\n",
                "\n",
                "# Calculer métriques\n",
                "f1_macro = f1_score(all_labels, all_predictions, average='macro', zero_division=0)\n",
                "f1_weighted = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)\n",
                "precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)\n",
                "recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)\n",
                "\n",
                "print(f'\\n✓ Test Set Performance:')\n",
                "print(f'  - Accuracy: {test_acc:.2f}%')\n",
                "print(f'  - F1 Macro: {f1_macro:.4f}')\n",
                "print(f'  - F1 Weighted: {f1_weighted:.4f}')\n",
                "print(f'  - Precision: {precision:.4f}')\n",
                "print(f'  - Recall: {recall:.4f}')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Sauvegarder modèle et résultats"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "print('Sauvegarde modèle et résultats...')\n",
                "\n",
                "# Sauvegarder modèle\n",
                "model_path = MODELS_PATH / 'resnet50_baseline.pth'\n",
                "torch.save(model.state_dict(), model_path)\n",
                "print(f'✓ Modèle: {model_path}')\n",
                "\n",
                "# Matrice de confusion\n",
                "cm = confusion_matrix(all_labels, all_predictions)\n",
                "cm_path = RESULTS_PATH / 'confusion_matrix_baseline.pkl'\n",
                "with open(cm_path, 'wb') as f:\n",
                "    pickle.dump(cm, f)\n",
                "print(f'✓ Confusion matrix: {cm_path}')\n",
                "\n",
                "# Historique entraînement\n",
                "history_path = RESULTS_PATH / 'training_history.json'\n",
                "with open(history_path, 'w') as f:\n",
                "    json.dump(history, f)\n",
                "print(f'✓ History: {history_path}')\n",
                "\n",
                "# Métriques\n",
                "metrics = {\n",
                "    'test_accuracy': float(test_acc),\n",
                "    'test_f1_macro': float(f1_macro),\n",
                "    'test_f1_weighted': float(f1_weighted),\n",
                "    'test_precision': float(precision),\n",
                "    'test_recall': float(recall),\n",
                "    'num_epochs_trained': len(history['train_loss'])\n",
                "}\n",
                "\n",
                "metrics_path = RESULTS_PATH / 'metrics_baseline.json'\n",
                "with open(metrics_path, 'w') as f:\n",
                "    json.dump(metrics, f, indent=2)\n",
                "print(f'✓ Metrics: {metrics_path}')\n",
                "\n",
                "print('\\n✓ Étape 2 complétée avec succès')"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## Visualiser training curves"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
                "\n",
                "# Loss\n",
                "axes[0].plot(history['train_loss'], label='Train', marker='o')\n",
                "axes[0].plot(history['val_loss'], label='Val', marker='o')\n",
                "axes[0].set_xlabel('Epoch')\n",
                "axes[0].set_ylabel('Loss')\n",
                "axes[0].set_title('Training Loss')\n",
                "axes[0].legend()\n",
                "axes[0].grid(True, alpha=0.3)\n",
                "\n",
                "# Accuracy\n",
                "axes[1].plot(history['train_acc'], label='Train', marker='o')\n",
                "axes[1].plot(history['val_acc'], label='Val', marker='o')\n",
                "axes[1].set_xlabel('Epoch')\n",
                "axes[1].set_ylabel('Accuracy (%)')\n",
                "axes[1].set_title('Training Accuracy')\n",
                "axes[1].legend()\n",
                "axes[1].grid(True, alpha=0.3)\n",
                "\n",
                "plt.tight_layout()\n",
                "curves_path = RESULTS_PATH / 'training_curves_baseline.png'\n",
                "plt.savefig(curves_path, dpi=150, bbox_inches='tight')\n",
                "plt.show()\n",
                "\n",
                "print(f'✓ Curves sauvegardées: {curves_path}')"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.10.0"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 4
}

# Sauvegarder le notebook
BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
NOTEBOOKS_PATH = BASE_PATH / 'notebooks'
NOTEBOOKS_PATH.mkdir(parents=True, exist_ok=True)

notebook_path = NOTEBOOKS_PATH / '02_preprocessing_training.ipynb'
with open(notebook_path, 'w') as f:
    json.dump(notebook_content, f, indent=2)

print(f"\n✓ Notebook créé et sauvegardé : {notebook_path}")
print(f"✓ Taille : {notebook_path.stat().st_size / 1024:.1f} KB")

# Vérifier
if notebook_path.exists():
    print(f"✓ Fichier existe et est valide")


✓ Notebook créé et sauvegardé : /content/drive/MyDrive/AnanthiX_AI/notebooks/02_preprocessing_training.ipynb
✓ Taille : 21.0 KB
✓ Fichier existe et est valide
